# AGENTS

<div align="center">
    <img src="images/agentllama.png" width="500" height="700">
</div>

| Agent             | Main idea                      | Typical use                                           |
| ----------------- | ------------------------------ | ----------------------------------------------------- |
| **FunctionAgent** | Uses LLM tool/function calling | General-purpose tool-using agents                     |
| **ReActAgent**    | Reason → Action → Observation  | Agents where explicit reasoning/action loop is useful |
| **CodeActAgent**  | Generates and executes code    | Data analysis, calculations, programmatic tasks       |
| **AgentWorkflow** | Coordinates multiple agents    | Multi-agent workflows                                 |


<h1 style="color: #16A34A;">1. FUNCTION AGENT</h1>

**LLM decides whether a tool is needed and which tool to call.**

<div align="center">
    <img src="images/functionagent.png" width="300" height="700">
</div>

In [1]:
import logging
logging.getLogger("opentelemetry").setLevel(logging.CRITICAL)
logging.getLogger("opentelemetry.exporter.otlp.proto.http.trace_exporter").setLevel(logging.CRITICAL)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [2]:
import asyncio
from dotenv import load_dotenv
from llama_index.llms.openai import OpenAI

load_dotenv()
llm = OpenAI(model="gpt-4o-mini", temperature=0)

In [4]:
from llama_index.core.agent.workflow import FunctionAgent

# Tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

# Agent
agent = FunctionAgent(
    tools=[add_numbers],
    llm=llm,
    system_prompt="You are a helpful mathematical assistant."
)

# Run Agent
async def main():
    response = await agent.run(user_msg="What is 25 + 15?")
    print("FINAL RESPONSE:")
    print(response)
await main()

FINAL RESPONSE:
25 + 15 equals 40.


<h1 style="color: #16A34A;">2. ReAct AGENT</h1>

**ReAct stands for --> Reason + Act**

<div align="center">
    <img src="images/reactagent.png" width="300" height="700">
</div>

In [5]:
from llama_index.core.agent.workflow import ReActAgent

# Tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

# Agent
agent = ReActAgent(
    tools=[multiply_numbers],
    llm=llm,
    system_prompt="You are a helpful mathematical assistant."
)

# Run
async def main():
    response = await agent.run(
        user_msg="What is 12 multiplied by 5?"
    )
    print("FINAL RESPONSE:")
    print(response)
await main()

FINAL RESPONSE:
12 multiplied by 5 is 60.


<h1 style="color: #16A34A;">3. CodeAct AGENT</h1>

**A CodeAct-style agent solves tasks by writing and executing code.**

<div align="center">
    <img src="images/codeagent.png" width="300" height="700">
</div>

In [6]:
from llama_index.core.agent.workflow import CodeActAgent

generated_code = []

async def code_execute_fn(code):
    generated_code.append(code)
    exec(code, globals())


# Agent
agent = CodeActAgent(
    tools=[],
    llm=llm,
    code_execute_fn=code_execute_fn,
    system_prompt="You are a helpful data analysis assistant."
)

# Run
async def main():

    handler = agent.run(user_msg="Calculate the average of 10, 20, 30, 40 and 50 using Python.")
    response = await handler
    print("\nGENERATED CODE / STEPS:")
    print("-" * 50)

    for code in generated_code:
        print(code)

    print("\nFINAL RESPONSE:")
    print(response)

await main()

The average is 30.0

GENERATED CODE / STEPS:
--------------------------------------------------
# Define the numbers
numbers = [10, 20, 30, 40, 50]

# Calculate the average
average = sum(numbers) / len(numbers)
print(f"The average is {average}")

FINAL RESPONSE:
The average of the numbers 10, 20, 30, 40, and 50 is 30.0.


<h1 style="color: #16A34A;">4. Agent Workflow</h1>

**AgentWorkflow is useful when you want multiple agents working together.**

<div align="center">
    <img src="images/agentworkflow.png" width="300" height="700">
</div>

In [7]:
from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent

# Agent 1
research_agent = ReActAgent(
    name="research_agent",
    description="Researches and analyzes the given topic.",
    system_prompt="""
    You are a research agent.
    Analyze the user's question and provide useful information.
    When your analysis is complete, hand off the task to writer_agent.
    """,
    tools=[],
    llm=llm,
    can_handoff_to=["writer_agent"]
)

# Agent 2
writer_agent = ReActAgent(
    name="writer_agent",
    description="Creates the final answer from the research.",
    system_prompt="""
    You are a writing agent.
    Take the information provided by the research agent
    and create a simple, clear final answer.
    """,
    tools=[],
    llm=llm
)

# Agent Workflow
workflow = AgentWorkflow(
    agents=[
        research_agent,
        writer_agent
    ],
    root_agent="research_agent"
)

# Run
async def main():
    response = await workflow.run(user_msg="Explain the benefits of artificial intelligence.")
    print("FINAL RESPONSE:")
    print(response)
await main()

FINAL RESPONSE:
Artificial intelligence (AI) offers numerous benefits across various sectors. Here are some key advantages:

1. **Efficiency and Automation**: AI can automate repetitive tasks, allowing humans to focus on more complex and creative work. This leads to increased productivity and efficiency in various industries.

2. **Data Analysis**: AI can process and analyze vast amounts of data quickly and accurately, uncovering insights that would be difficult for humans to identify. This capability is invaluable in fields like finance, healthcare, and marketing.

3. **Personalization**: AI enables personalized experiences for users by analyzing their preferences and behaviors. This is evident in applications like recommendation systems used by streaming services and e-commerce platforms.

4. **Improved Decision-Making**: AI systems can provide data-driven insights that help organizations make informed decisions, reducing the risk of human error and bias.

5. **Enhanced Customer Serv